# Prototipazione del Data Agent — Risk & Credit Intelligence Hub

Questo notebook documenta il lavoro di prototipazione svolto **prima** di integrare la logica di data cleaning, analisi e visualizzazione nell'architettura finale (`data_agent/agent_engine.py` + `data_agent/main.py`).

Obiettivo: validare in modo interattivo, con output visibile passo-passo, il comportamento probabilistico/deterministico del motore di analisi prima di esporlo come microservizio FastAPI consumato dal Backend Node.js.

**Dataset**: `database/intesa_core_banking.db` — 4 tabelle relazionali (T_CLIENTI, T_PRATICHE_FIDO, T_FILIALI, T_PERFORMANCE_AMORT), derivate dal CSV originale del progetto, con un 10-12% di record volutamente "sporchi" (valori mancanti, formati incoerenti, outlier).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DB_PATH = "../data_agent/database/intesa_core_banking.db"
sns.set_theme(style="whitegrid")

conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables

## 1. Ispezione dei dati grezzi

Prima regola della prototipazione: **non fidarsi mai del dato grezzo**. Carichiamo `T_CLIENTI` senza alcuna pulizia e osserviamo i problemi reali che il Data Agent dovrà gestire in autonomia.

In [ ]:
raw_clienti = pd.read_sql_query("SELECT * FROM T_CLIENTI", conn)

print(f"Righe totali: {len(raw_clienti)}")
print(f"Valori nulli su Reddito_Annuale_EUR: {raw_clienti['Reddito_Annuale_EUR'].isna().sum()}")
print(f"Righe duplicate (ID_Cliente): {raw_clienti['ID_Cliente'].duplicated().sum()}")
print()
print("Esempio di formati incoerenti nella colonna Reddito_Annuale_EUR:")
raw_clienti['Reddito_Annuale_EUR'].astype(str).sample(8, random_state=42)

## 2. Prototipazione della pipeline di Data Cleaning

Questa è la logica che, una volta validata qui, è stata trasferita **identica** in `DataAgentEngine.clean_dataset()` dentro `data_agent/agent_engine.py`.

In [ ]:
def clean_dataset_prototype(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Stripping delle stringhe (spazi superflui in categorie professionali, ecc.)
    str_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()

    # Normalizzazione Reddito_Annuale_EUR: gestisce sia "45.000 \u20ac" sia numeri puliti
    if 'Reddito_Annuale_EUR' in df.columns:
        df['Reddito_Annuale_EUR'] = (
            df['Reddito_Annuale_EUR']
            .astype(str)
            .str.replace('\u20ac', '', regex=False)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df['Reddito_Annuale_EUR'] = pd.to_numeric(df['Reddito_Annuale_EUR'], errors='coerce')
        # Imputazione dei nulli con la mediana, invece di scartare le righe
        df['Reddito_Annuale_EUR'] = df['Reddito_Annuale_EUR'].fillna(df['Reddito_Annuale_EUR'].median())

    # Eta_Cliente: stesso trattamento
    if 'Eta_Cliente' in df.columns:
        df['Eta_Cliente'] = pd.to_numeric(df['Eta_Cliente'], errors='coerce')
        df['Eta_Cliente'] = df['Eta_Cliente'].fillna(df['Eta_Cliente'].median())

    # Deduplicazione su ID_Cliente, mantenendo la prima occorrenza
    if 'ID_Cliente' in df.columns:
        df = df.drop_duplicates(subset=['ID_Cliente'], keep='first')

    return df

cleaned_clienti = clean_dataset_prototype(raw_clienti)

print(f"Righe dopo cleaning: {len(cleaned_clienti)} (da {len(raw_clienti)})")
print(f"Valori nulli residui su Reddito_Annuale_EUR: {cleaned_clienti['Reddito_Annuale_EUR'].isna().sum()}")
cleaned_clienti[['ID_Cliente', 'Eta_Cliente', 'Reddito_Annuale_EUR']].describe()

## 3. Prototipazione delle query analitiche cross-tabella

Le due query qui sotto sono le stesse (a meno di formattazione) poi codificate come esempi vincolanti nel prompt del ReAct Agent (`backend/src/agent/reactAgent.js`, REGOLE 1 e 1b), proprio perché in fase di prototipazione qui si è scoperto che l'LLM tendeva a sbagliare il JOIN con `T_PERFORMANCE_AMORT` per il calcolo del tasso di default.

In [ ]:
query_esposizione = """
SELECT f.Nome_Filiale, SUM(p.Importo_Richiesto_EUR) as Somma_Fidi
FROM T_PRATICHE_FIDO p
JOIN T_CLIENTI c ON p.ID_Cliente = c.ID_Cliente
JOIN T_FILIALI f ON c.Filiale_ID = f.Filiale_ID
GROUP BY f.Nome_Filiale
ORDER BY Somma_Fidi DESC
"""
df_esposizione = pd.read_sql_query(query_esposizione, conn)
df_esposizione

In [ ]:
query_default_rate = """
SELECT f.Nome_Filiale,
       COUNT(DISTINCT p.ID_Pratica) as Totale_Pratiche,
       SUM(CASE WHEN pa.Flag_Default_12M = 1 THEN 1 ELSE 0 END) as Pratiche_Default,
       ROUND(SUM(CASE WHEN pa.Flag_Default_12M = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT p.ID_Pratica), 2) as Tasso_Default_Pct
FROM T_PRATICHE_FIDO p
JOIN T_CLIENTI c ON p.ID_Cliente = c.ID_Cliente
JOIN T_FILIALI f ON c.Filiale_ID = f.Filiale_ID
JOIN T_PERFORMANCE_AMORT pa ON pa.ID_Pratica = p.ID_Pratica
GROUP BY f.Nome_Filiale
ORDER BY Tasso_Default_Pct DESC
"""
df_default = pd.read_sql_query(query_default_rate, conn)
df_default

## 4. Prototipazione della generazione grafici (Seaborn)

Stessa logica di plotting poi trasferita in `DataAgentEngine.generate_chart()`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df_default, x='Nome_Filiale', y='Tasso_Default_Pct', color='#005A9C', ax=ax)
ax.set_title('Tasso di Default a 12 mesi per Filiale', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Tasso di Default (%)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 5. Prototipazione della sintesi narrativa

Traduzione dei numeri grezzi in un insight testuale comprensibile per un decision maker — stessa logica poi codificata in `DataAgentEngine.generate_summary()`.

In [ ]:
def generate_summary_prototype(df: pd.DataFrame) -> str:
    top = df.iloc[0]
    bottom = df.iloc[-1]
    media = df['Tasso_Default_Pct'].mean()
    return (
        f"Il tasso di default medio sulle {len(df)} filiali analizzate e' del {media:.2f}%. "
        f"La filiale piu' a rischio e' '{top['Nome_Filiale']}' ({top['Tasso_Default_Pct']}%), "
        f"mentre la piu' virtuosa e' '{bottom['Nome_Filiale']}' ({bottom['Tasso_Default_Pct']}%)."
    )

print(generate_summary_prototype(df_default))

conn.close()

## Conclusioni della prototipazione

- La pipeline di cleaning gestisce correttamente valori mancanti, formati incoerenti ("45.000 \u20ac") e duplicati, senza scartare righe quando evitabile (imputazione con mediana).
- Il JOIN su `T_PERFORMANCE_AMORT` per il tasso di default **non è ovvio** per un LLM: da qui la scelta di codificarlo come esempio esplicito nel prompt dell'agente, invece di lasciare che l'LLM lo scopra da solo ad ogni richiesta (più lento, meno affidabile).
- Grafici e sintesi narrativa prodotti qui sono stati trasferiti 1:1 nel microservizio FastAPI (`data_agent/main.py`), che li espone come endpoint richiamato dal Backend Node.js tramite il tool `execute_data_analytics`.